# MLP Experience Replay
Train an MLPClassifier with year-wise incremental scaling and experience replay.

In [2]:
import copy
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')

## 1. Load Data and Splits

In [3]:
ds_path = Path('.') / 'training_data_with_features_plus_monthly_indices.zarr'
print(f'Loading data from {ds_path}...')
ds = xr.open_dataset(ds_path, engine='zarr')
print('Data loaded')

split_path = Path('.') / 'data_split.npz'
print(f'Loading split from {split_path}...')
split_data = np.load(split_path)
train_pixel_indices = split_data['train_pixel_indices']
val_pixel_indices = split_data['val_pixel_indices']
test_pixel_indices = split_data['test_pixel_indices']
print('Split loaded')

print('Dataset info:')
print(f'  Total pixels: {len(ds.pixel)}')
print(f'  Total years: {len(ds.year)}')
print(f'  Train pixels: {len(train_pixel_indices)}')
print(f'  Val pixels: {len(val_pixel_indices)}')
print(f'  Test pixels: {len(test_pixel_indices)}')

Loading data from training_data_with_features_plus_monthly_indices.zarr...
Data loaded
Loading split from data_split.npz...
Split loaded
Dataset info:
  Total pixels: 8155205
  Total years: 7
  Train pixels: 5597776
  Val pixels: 1273437
  Test pixels: 1283992


## 2. Feature Engineering

In [4]:
def prepare_raw_features_for_year(ds, pixel_indices, year_idx, s2_mean_per_pixel=None, dtype=np.float32):
    """Extract and clean raw (unscaled) features for one year. Year 0 is skipped by design."""
    if year_idx == 0:
        return np.empty((0, 0), dtype=dtype), np.empty((0,), dtype=np.int64)

    if s2_mean_per_pixel is None:
        s2_all_years = ds['s2_bands'].isel(pixel=pixel_indices).values
        s2_mean_per_pixel = np.nanmean(s2_all_years, axis=1)

    ds_subset = ds.isel(pixel=pixel_indices, year=year_idx)
    s2_features = ds_subset['s2_bands'].values
    if np.isnan(s2_features).any():
        s2_features = np.where(np.isnan(s2_features), s2_mean_per_pixel, s2_features)

    dem_features = ds_subset['dem'].values.reshape(-1, 1)
    ndvi_features = ds_subset['ndvi'].values.reshape(-1, 1)
    ndwi_features = ds_subset['ndwi'].values.reshape(-1, 1)

    ds_prev = ds.isel(pixel=pixel_indices, year=year_idx - 1)
    ndvi_last_year = np.where(np.isnan(ds_prev['ndvi'].values.reshape(-1, 1)), 0, ds_prev['ndvi'].values.reshape(-1, 1))
    ndwi_last_year = np.where(np.isnan(ds_prev['ndwi'].values.reshape(-1, 1)), 0, ds_prev['ndwi'].values.reshape(-1, 1))

    required_last_year_bands = ['B04', 'B03', 'B06']
    band_to_idx = {band: i for i, band in enumerate(ds['s2_band'].values)}
    missing_bands = [band for band in required_last_year_bands if band not in band_to_idx]
    if missing_bands:
        raise ValueError(f"Missing required S2 bands for last-year features: {missing_bands}")

    last_year_s2_features = []
    for band in required_last_year_bands:
        band_values = ds_prev['s2_bands'].sel(s2_band=band).values
        band_values = np.where(np.isnan(band_values), 0, band_values)
        last_year_s2_features.append(band_values.reshape(-1, 1))

    # Keep feature order aligned with MLP training notebook.
    features_list = [
        s2_features,
        dem_features,
        ndvi_features,
        ndwi_features,
        ndvi_last_year,
        ndwi_last_year,
        *last_year_s2_features,
    ]

    if 'nbr' in ds.data_vars:
        features_list.append(ds_subset['nbr'].values.reshape(-1, 1))

    if year_idx > 0 and 'ndvi_delta' in ds.data_vars:
        delta_year_idx = year_idx - 1
        ds_delta = ds.isel(pixel=pixel_indices, year=delta_year_idx)
        features_list.append(ds_delta['ndvi_delta'].values.reshape(-1, 1))
        features_list.append(ds_delta['ndwi_delta'].values.reshape(-1, 1))
        if 'nbr_delta' in ds.data_vars:
            features_list.append(ds_delta['nbr_delta'].values.reshape(-1, 1))

    for var_name in ['years_since_last_disturbance', 'log_years_since_last_disturbance', 'ever_disturbed']:
        if var_name in ds.data_vars:
            features_list.append(ds_subset[var_name].values.reshape(-1, 1))

    yearly_index_feature_names = [
        'ndvi_cv_year',
        'ndvi_max_m2m_drop_year',
        'ndvi_max_year',
        'ndvi_min_year',
        'ndvi_std_year',
        'ndwi_cv_year',
        'ndwi_max_m2m_drop_year',
        'ndwi_max_year',
        'ndwi_min_year',
        'ndwi_std_year',
    ]
    for feature_name in yearly_index_feature_names:
        if feature_name in ds.data_vars:
            features_list.append(ds_subset[feature_name].values.reshape(-1, 1))

    X = np.concatenate(features_list, axis=1)
    y = ds_subset['disturbances'].values

    valid_label_mask = np.isin(y, [0, 1])
    X = X[valid_label_mask]
    y = y[valid_label_mask]

    nan_mask = ~np.isnan(X).any(axis=1)
    X_clean = X[nan_mask]
    y_clean = y[nan_mask]

    if len(X_clean) == 0:
        feature_dim = X.shape[1] if X.ndim == 2 and X.shape[0] > 0 else 0
        return np.empty((0, feature_dim), dtype=dtype), np.empty((0,), dtype=np.int64)

    return X_clean.astype(dtype, copy=False), y_clean.astype(np.int64, copy=False)


def prepare_features_for_year(ds, pixel_indices, year_idx, scaler=None, scaler_mode='auto', s2_mean_per_pixel=None, dtype=np.float32):
    """Extract, clean, and optionally scale features for one year. Year 0 is skipped by design."""
    valid_scaler_modes = {'auto', 'fit', 'partial_fit', 'transform', 'none'}
    if scaler_mode not in valid_scaler_modes:
        raise ValueError(f"Invalid scaler_mode '{scaler_mode}'. Valid options: {sorted(valid_scaler_modes)}")

    X_clean, y_clean = prepare_raw_features_for_year(
        ds,
        pixel_indices,
        year_idx,
        s2_mean_per_pixel=s2_mean_per_pixel,
        dtype=dtype,
    )

    if len(X_clean) == 0:
        return X_clean, y_clean, scaler

    if scaler_mode == 'none':
        return X_clean, y_clean, scaler

    if scaler is None:
        scaler = StandardScaler()

    if scaler_mode == 'auto':
        if hasattr(scaler, 'mean_'):
            X_clean = scaler.transform(X_clean)
        else:
            X_clean = scaler.fit_transform(X_clean)
    elif scaler_mode == 'fit':
        X_clean = scaler.fit_transform(X_clean)
    elif scaler_mode == 'partial_fit':
        scaler.partial_fit(X_clean)
        X_clean = scaler.transform(X_clean)
    elif scaler_mode == 'transform':
        if not hasattr(scaler, 'mean_'):
            raise ValueError("Scaler must be fitted before using scaler_mode='transform'.")
        X_clean = scaler.transform(X_clean)

    return X_clean, y_clean, scaler

## 3. Initialize Class Weights and MLP

In [5]:
print('Precomputing raw yearly features for train/validation splits...')
n_years = len(ds.year)

def precompute_yearly_raw_cache(ds, pixel_indices, n_years, split_name):
    s2_all_years = ds['s2_bands'].isel(pixel=pixel_indices).values
    s2_mean_per_pixel = np.nanmean(s2_all_years, axis=1)

    cache = {}
    empty_years = 0
    for year_idx in tqdm(range(1, n_years), desc=f'Precompute {split_name}'):
        X_raw, y_raw = prepare_raw_features_for_year(
            ds,
            pixel_indices,
            year_idx,
            s2_mean_per_pixel=s2_mean_per_pixel,
            dtype=np.float32,
        )
        cache[year_idx] = (X_raw, y_raw)
        if len(y_raw) == 0:
            empty_years += 1

    print(
        f"{split_name}: cached {len(cache)} years, empty years={empty_years}, "
        f"sample feature dim={next((x.shape[1] for x, y in cache.values() if len(y) > 0), 0)}"
    )
    return cache

train_feature_cache = precompute_yearly_raw_cache(ds, train_pixel_indices, n_years, 'train')
val_feature_cache = precompute_yearly_raw_cache(ds, val_pixel_indices, n_years, 'validation')

print('Computing class weights from cached training labels...')
train_label_batches = [y_batch for _, y_batch in train_feature_cache.values() if len(y_batch) > 0]
if not train_label_batches:
    raise ValueError('No valid training labels after filtering.')

all_train_labels = np.concatenate(train_label_batches).astype(int)
all_train_labels = all_train_labels[np.isin(all_train_labels, [0, 1])]
if len(all_train_labels) == 0:
    raise ValueError('No valid training labels after filtering.')

classes = np.array([0, 1])
class_weights_array = compute_class_weight('balanced', classes=classes, y=all_train_labels)
class_weight_dict = {classes[i]: class_weights_array[i] for i in range(len(classes))}

print('Class weights:')
print(f"  Class 0: {class_weight_dict[0]:.4f}")
print(f"  Class 1: {class_weight_dict[1]:.4f}")

model = MLPClassifier(
    hidden_layer_sizes=(64,),
    activation='relu',
    alpha=0.0001,
    random_state=42,
    solver='adam',
    learning_rate='adaptive',
    max_iter=1,
    learning_rate_init=0.001,
    warm_start=False,
    verbose=False,
)
print('MLP initialized')

Precomputing raw yearly features for train/validation splits...


Precompute train:   0%|          | 0/6 [00:00<?, ?it/s]

train: cached 6 years, empty years=0, sample feature dim=32


Precompute validation:   0%|          | 0/6 [00:00<?, ?it/s]

validation: cached 6 years, empty years=0, sample feature dim=32
Computing class weights from cached training labels...
Class weights:
  Class 0: 0.5105
  Class 1: 24.2384
MLP initialized


## 4. Replay-Enabled Online Training

In [ ]:
import json
from datetime import datetime

REPLAY_RATIOS = [0.2, 0.3, 0.4, 0.5]
REPLAY_ENABLED = True
REPLAY_RANDOM_STATE = 42
TARGET_REPLAY_POSITIVE_RATE = 0.15
REPLAY_POSITIVE_LABEL = 1
TARGET_POSITIVE_RATE_TOKEN = f'PR_{TARGET_REPLAY_POSITIVE_RATE:.2f}'

# Origin-aware weighting policy:
# current-year weights are computed from current-year labels; replay weights are computed after replay sampling.
WEIGHT_POLICY_NAME = 'post_sampling_replay_class_weights_v1'
REPLAY_WEIGHT_FALLBACK = 'smoothed_single_class'
STRICT_POLICY_RESUME_CHECK = True

CHUNK_SIZE = 50000
MAX_EPOCHS = 15
PATIENCE = 3
MIN_DELTA = 0.0005

if not 0.0 < TARGET_REPLAY_POSITIVE_RATE <= 1.0:
    raise ValueError(f'TARGET_REPLAY_POSITIVE_RATE must be in (0, 1], got {TARGET_REPLAY_POSITIVE_RATE}')
if REPLAY_WEIGHT_FALLBACK not in {'smoothed_single_class'}:
    raise ValueError(f'Unsupported REPLAY_WEIGHT_FALLBACK: {REPLAY_WEIGHT_FALLBACK}')

CHECKPOINT_DIR = Path('.') / f'training_checkpoints_mlp_experience_replay_{TARGET_POSITIVE_RATE_TOKEN}'
ALL_HISTORIES_FILE = CHECKPOINT_DIR / 'mlp_replay_all_training_histories.pkl'
COMPLETION_STATUS_FILE = CHECKPOINT_DIR / 'mlp_replay_completion_status.json'
TRAINING_LOG_FILE = CHECKPOINT_DIR / 'mlp_replay_training_log.txt'
COMBINED_HISTORY_FILENAME = f'mlp_classifier_history_prevyears_monthly_features_incremental_scaler_experience_replay_{TARGET_POSITIVE_RATE_TOKEN}_all_ratios.csv'

CHECKPOINT_DIR.mkdir(exist_ok=True)

if 'train_feature_cache' not in globals() or 'val_feature_cache' not in globals():
    raise ValueError('Raw feature caches not found. Run Cell 8 first to precompute yearly features.')

def create_empty_training_history():
    return {
        'year': [],
        'train_accuracy': [],
        'train_precision': [],
        'train_recall': [],
        'train_f1': [],
        'val_accuracy': [],
        'val_precision': [],
        'val_recall': [],
        'val_f1': [],
        'val_roc_auc': [],
        'val_pr_auc': [],
        'replay_pool_size': [],
        'replay_target_size': [],
        'replay_used_size': [],
        'replay_target_positive_rate': [],
        'replay_actual_positive_rate': [],
        'weight_policy': [],
        'current_origin_weight': [],
        'year_positive_rate': [],
        'target_positive_rate': [],
        'replay_multiplier_unclipped': [],
        'replay_multiplier': [],
        'replay_pool_positive_rate': [],
        'current_class_weight_0': [],
        'current_class_weight_1': [],
        'replay_class_weight_0': [],
        'replay_class_weight_1': [],
        'replay_weight_fallback': [],
    }

def load_all_training_histories(path):
    if path.exists():
        with open(path, 'rb') as f:
            data = pickle.load(f)
        if isinstance(data, dict):
            return data
    return {}

def save_all_training_histories(path, all_histories):
    with open(path, 'wb') as f:
        pickle.dump(all_histories, f)

def load_completion_status(path):
    default_status = {'completed_ratios': [], 'completed_years': {}, 'weight_policy_by_ratio': {}}
    if path.exists():
        with open(path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        if isinstance(data, dict):
            if 'completed_ratios' not in data:
                data['completed_ratios'] = []
            if 'completed_years' not in data:
                data['completed_years'] = {}
            if 'weight_policy_by_ratio' not in data:
                data['weight_policy_by_ratio'] = {}
            return data
    return default_status

def save_completion_status(path, status):
    status['completed_ratios'] = sorted(list(set(status.get('completed_ratios', []))))
    cleaned_completed_years = {}
    for key, years in status.get('completed_years', {}).items():
        cleaned_completed_years[key] = sorted(list(set(int(y) for y in years)))
    status['completed_years'] = cleaned_completed_years

    with open(path, 'w', encoding='utf-8') as f:
        json.dump(status, f, indent=2)

def append_training_log(path, message):
    timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    with open(path, 'a', encoding='utf-8') as f:
        f.write(f'[{timestamp}] {message}\n')

def format_ratio_key(replay_ratio):
    return f'RR_{replay_ratio:.1f}'

def build_mlp_model():
    return MLPClassifier(
        hidden_layer_sizes=(64,),
        activation='relu',
        alpha=0.0001,
        random_state=42,
        solver='adam',
        learning_rate='adaptive',
        max_iter=1,
        learning_rate_init=0.001,
        warm_start=False,
        verbose=False,
    )

def get_cached_raw_year(cache, year_idx):
    if year_idx not in cache:
        return np.empty((0, 0), dtype=np.float32), np.empty((0,), dtype=np.int64)
    return cache[year_idx]

def build_replay_year_metadata(cache, current_year_idx, positive_label=1):
    metadata = []
    replay_pool_size = 0
    replay_positive_pool_size = 0
    replay_negative_pool_size = 0

    for past_year_idx in range(1, current_year_idx):
        X_past_raw, y_past = get_cached_raw_year(cache, past_year_idx)
        if len(X_past_raw) == 0:
            continue

        unique_replay_labels = set(np.unique(y_past).tolist())
        if not unique_replay_labels.issubset({0, 1}):
            raise ValueError(f'Expected binary replay labels in {{0, 1}}, got {sorted(unique_replay_labels)}')

        positive_indices = np.flatnonzero(y_past == positive_label)
        negative_indices = np.flatnonzero(y_past != positive_label)

        metadata.append(
            {
                'year_idx': past_year_idx,
                'positive_indices': positive_indices,
                'negative_indices': negative_indices,
            }
        )

        replay_pool_size += len(y_past)
        replay_positive_pool_size += len(positive_indices)
        replay_negative_pool_size += len(negative_indices)

    return metadata, replay_pool_size, replay_positive_pool_size, replay_negative_pool_size

def sample_grouped_indices(sources, sample_size, rng):
    if sample_size <= 0 or not sources:
        return []

    source_counts = np.array([len(indices) for _, indices in sources], dtype=np.int64)
    total_count = int(source_counts.sum())
    if total_count == 0:
        return []

    sample_size = int(min(sample_size, total_count))
    picked_global = np.sort(rng.choice(total_count, size=sample_size, replace=False))

    cumulative = np.cumsum(source_counts)
    source_ids = np.searchsorted(cumulative, picked_global, side='right')
    cumulative_prev = np.concatenate(([0], cumulative[:-1]))

    grouped = []
    for source_id in np.unique(source_ids):
        source_id_int = int(source_id)
        source_mask = source_ids == source_id_int
        local_offsets = picked_global[source_mask] - cumulative_prev[source_id_int]
        year_idx, year_local_indices = sources[source_id_int]
        sampled_local_indices = year_local_indices[local_offsets]
        grouped.append((year_idx, sampled_local_indices))

    return grouped

def materialize_replay_samples(grouped_indices, cache, scaler):
    if not grouped_indices:
        return np.empty((0, 0), dtype=np.float32), np.empty((0,), dtype=np.int64)

    replay_X_chunks = []
    replay_y_chunks = []
    feature_dim = 0

    for past_year_idx, sampled_local_indices in grouped_indices:
        if len(sampled_local_indices) == 0:
            continue

        X_past_raw, y_past = get_cached_raw_year(cache, past_year_idx)
        if len(X_past_raw) == 0:
            continue

        X_sampled_raw = X_past_raw[sampled_local_indices]
        y_sampled = y_past[sampled_local_indices]

        X_sampled_scaled = scaler.transform(X_sampled_raw)
        replay_X_chunks.append(X_sampled_scaled)
        replay_y_chunks.append(y_sampled)
        feature_dim = X_sampled_scaled.shape[1]

    if not replay_X_chunks:
        return np.empty((0, feature_dim), dtype=np.float32), np.empty((0,), dtype=np.int64)

    X_replay_sampled = np.vstack(replay_X_chunks)
    y_replay_sampled = np.concatenate(replay_y_chunks)
    return X_replay_sampled, y_replay_sampled

def compute_year_positive_rate(y_batch, positive_label=1):
    if len(y_batch) == 0:
        return np.nan
    return float(np.mean(y_batch == positive_label))

def compute_binary_class_weights(y_batch, classes=np.array([0, 1]), fallback_mode='smoothed_single_class'):
    y_batch = np.asarray(y_batch, dtype=np.int64)
    if len(y_batch) == 0:
        return {0: 1.0, 1: 1.0}, 'empty_uniform'

    unique_labels = set(np.unique(y_batch).tolist())
    if not unique_labels.issubset({0, 1}):
        raise ValueError(f'Expected binary labels in {{0, 1}}, got {sorted(unique_labels)}')

    if len(unique_labels) == 2:
        class_weights_array = compute_class_weight('balanced', classes=classes, y=y_batch)
        class_weight_dict = {classes[i]: float(class_weights_array[i]) for i in range(len(classes))}
        return class_weight_dict, 'balanced'

    if fallback_mode != 'smoothed_single_class':
        raise ValueError(f'Unsupported fallback_mode: {fallback_mode}')

    # Laplace-smoothed prevalence keeps weights finite even when one class is absent.
    n_samples = float(len(y_batch))
    positive_count = float(np.sum(y_batch == 1))
    positive_rate_smoothed = (positive_count + 1.0) / (n_samples + 2.0)
    class_weight_1 = 1.0 / (2.0 * positive_rate_smoothed)
    class_weight_0 = 1.0 / (2.0 * (1.0 - positive_rate_smoothed))
    class_weight_dict = {0: float(class_weight_0), 1: float(class_weight_1)}

    present_label = int(next(iter(unique_labels)))
    return class_weight_dict, f'smoothed_single_class_present_{present_label}'

all_training_histories = load_all_training_histories(ALL_HISTORIES_FILE)
completion_status = load_completion_status(COMPLETION_STATUS_FILE)
completion_status.setdefault('weight_policy_by_ratio', {})

n_years = len(ds.year)
year_values = ds.year.values
year_value_to_idx = {int(y): idx for idx, y in enumerate(year_values)}
all_target_year_values = [int(year_values[idx]) for idx in range(1, n_years)]

print(f'Checkpoint directory: {CHECKPOINT_DIR.resolve()}')
print(f'Replay ratios: {REPLAY_RATIOS}')
print(f'Replay target positive rate: {TARGET_REPLAY_POSITIVE_RATE:.2%} ({TARGET_POSITIVE_RATE_TOKEN})')
print(
    f'Weight policy: {WEIGHT_POLICY_NAME}, '
    f'replay_weight_fallback={REPLAY_WEIGHT_FALLBACK}'
)
append_training_log(TRAINING_LOG_FILE, f'Started training run for ratios: {REPLAY_RATIOS}')

for ratio_idx, replay_ratio in enumerate(REPLAY_RATIOS):
    ratio_key = format_ratio_key(replay_ratio)
    output_suffix = f'incremental_scaler_experience_replay_{ratio_key}_{TARGET_POSITIVE_RATE_TOKEN}'
    models_dir_name = f'models_mlp_prevyears_monthly_features_{output_suffix}'
    scaler_file_template = f'scaler_year_{{year}}_mlp_prevyears_monthly_features_{output_suffix}.pkl'
    model_file_template = f'model_year_{{year}}_mlp_prevyears_monthly_features_{output_suffix}.pkl'
    final_scaler_filename = f'scaler_final_mlp_prevyears_monthly_features_{output_suffix}.pkl'
    final_model_filename = f'mlp_classifier_model_prevyears_monthly_features_{output_suffix}.pkl'
    history_filename = f'mlp_classifier_history_prevyears_monthly_features_{output_suffix}.csv'

    models_dir = Path('.') / models_dir_name
    models_dir.mkdir(exist_ok=True)

    expected_policy_config = {
        'weight_policy': WEIGHT_POLICY_NAME,
        'replay_weight_fallback': REPLAY_WEIGHT_FALLBACK,
        'target_positive_rate': float(TARGET_REPLAY_POSITIVE_RATE),
        'weight_source': 'current_year_and_post_sampling_replay',
    }
    existing_policy_config = completion_status.get('weight_policy_by_ratio', {}).get(ratio_key)
    if existing_policy_config is not None and existing_policy_config != expected_policy_config:
        policy_msg = (
            f'[{ratio_key}] Stored weight policy does not match current config. '
            f'Stored={existing_policy_config}, Current={expected_policy_config}'
        )
        if STRICT_POLICY_RESUME_CHECK:
            raise ValueError(policy_msg)
        print(f'WARNING: {policy_msg}')
        append_training_log(TRAINING_LOG_FILE, f'WARNING: {policy_msg}')

    completion_status['weight_policy_by_ratio'][ratio_key] = expected_policy_config

    if ratio_key in completion_status.get('completed_ratios', []):
        print(f'[{ratio_key}] already completed. Skipping ratio.')
        append_training_log(TRAINING_LOG_FILE, f'[{ratio_key}] skipped (already completed).')
        continue

    training_history = all_training_histories.get(ratio_key, create_empty_training_history())
    for key in create_empty_training_history().keys():
        training_history.setdefault(key, [])

    completed_years = set(int(y) for y in completion_status.get('completed_years', {}).get(ratio_key, []))
    completed_years.update(int(y) for y in training_history.get('year', []))
    completion_status.setdefault('completed_years', {})[ratio_key] = sorted(list(completed_years))

    replay_rng = np.random.default_rng(REPLAY_RANDOM_STATE + ratio_idx)

    model = None
    incremental_scaler = None
    start_year_idx = 1

    if completed_years:
        resume_candidate_years = sorted(completed_years, reverse=True)
        resumed = False
        for resume_year in resume_candidate_years:
            year_model_path = models_dir / model_file_template.format(year=resume_year)
            year_scaler_path = models_dir / scaler_file_template.format(year=resume_year)
            if year_model_path.exists() and year_scaler_path.exists() and resume_year in year_value_to_idx:
                with open(year_model_path, 'rb') as f:
                    model = pickle.load(f)
                with open(year_scaler_path, 'rb') as f:
                    incremental_scaler = pickle.load(f)
                start_year_idx = year_value_to_idx[resume_year] + 1
                resumed = True
                print(f'[{ratio_key}] resuming from year {resume_year}; continuing at index {start_year_idx}.')
                append_training_log(TRAINING_LOG_FILE, f'[{ratio_key}] resumed from year {resume_year}.')
                break

        if not resumed:
            print(f'[{ratio_key}] checkpoint artifacts missing/inconsistent. Restarting this ratio from scratch.')
            append_training_log(TRAINING_LOG_FILE, f'[{ratio_key}] restart due to missing/inconsistent checkpoint artifacts.')
            completed_years = set()
            completion_status['completed_years'][ratio_key] = []
            training_history = create_empty_training_history()

    if model is None:
        model = build_mlp_model()
    if incremental_scaler is None:
        incremental_scaler = StandardScaler()

    print(f'[{ratio_key}] Model output directory: {models_dir.resolve()}')
    print(f'[{ratio_key}] Years to train: 1..{n_years - 1} (year 0 skipped)')

    for year_idx in tqdm(range(start_year_idx, n_years), desc=f'{ratio_key} by year'):
        year_val = int(year_values[year_idx])

        if year_val in completed_years:
            continue

        X_train_raw, y_train_batch = get_cached_raw_year(train_feature_cache, year_idx)
        X_val_raw, y_val_batch = get_cached_raw_year(val_feature_cache, year_idx)

        if len(X_train_raw) == 0 or len(X_val_raw) == 0:
            print(f'[{ratio_key}] Year {year_val}: skipped (empty after filtering)')
            completed_years.add(year_val)
            completion_status['completed_years'][ratio_key] = sorted(list(completed_years))
            save_completion_status(COMPLETION_STATUS_FILE, completion_status)
            append_training_log(TRAINING_LOG_FILE, f'[{ratio_key}] year {year_val} skipped (empty after filtering).')
            continue

        year_positive_rate = compute_year_positive_rate(y_train_batch, positive_label=REPLAY_POSITIVE_LABEL)
        current_class_weight_dict, current_weight_fallback = compute_binary_class_weights(
            y_train_batch,
            classes=classes,
            fallback_mode=REPLAY_WEIGHT_FALLBACK,
        )

        incremental_scaler.partial_fit(X_train_raw)
        X_train_batch = incremental_scaler.transform(X_train_raw)
        X_val_batch = incremental_scaler.transform(X_val_raw)

        scaler_checkpoint = copy.deepcopy(incremental_scaler)
        year_scaler_path = models_dir / scaler_file_template.format(year=year_val)
        with open(year_scaler_path, 'wb') as f:
            pickle.dump(scaler_checkpoint, f)

        n_samples = len(X_train_batch)
        replay_target_size = int(n_samples * replay_ratio) if REPLAY_ENABLED else 0

        replay_metadata, replay_pool_size, replay_positive_pool_size, replay_negative_pool_size = (
            build_replay_year_metadata(train_feature_cache, year_idx, positive_label=REPLAY_POSITIVE_LABEL)
            if REPLAY_ENABLED and year_idx > 1
            else ([], 0, 0, 0)
        )

        replay_used_size = min(replay_target_size, replay_pool_size) if REPLAY_ENABLED else 0
        replay_actual_positive_rate = np.nan
        replay_pool_positive_rate = (
            float(replay_positive_pool_size / replay_pool_size)
            if replay_pool_size > 0
            else np.nan
        )

        positive_sources = [(item['year_idx'], item['positive_indices']) for item in replay_metadata if len(item['positive_indices']) > 0]
        negative_sources = [(item['year_idx'], item['negative_indices']) for item in replay_metadata if len(item['negative_indices']) > 0]

        if hasattr(model, 'n_features_in_') and int(model.n_features_in_) != int(X_train_batch.shape[1]):
            raise ValueError(
                f'Feature count mismatch at year {year_val}: model expects {model.n_features_in_}, got {X_train_batch.shape[1]}'
            )

        best_val_pr_auc = -np.inf
        patience_counter = 0
        best_model_state = None

        current_effective_mean_weight = np.nan
        replay_effective_mean_weight = np.nan
        replay_class_weight_dict = {0: np.nan, 1: np.nan}
        replay_weight_fallback = 'no_replay'

        for epoch in range(MAX_EPOCHS):
            if replay_used_size > 0:
                desired_positive_size = int(round(replay_used_size * TARGET_REPLAY_POSITIVE_RATE))
                desired_positive_size = min(max(desired_positive_size, 0), replay_used_size)

                sampled_positive_size = min(desired_positive_size, replay_positive_pool_size)
                sampled_negative_size = min(replay_used_size - sampled_positive_size, replay_negative_pool_size)

                remaining_to_fill = replay_used_size - sampled_positive_size - sampled_negative_size
                if remaining_to_fill > 0 and replay_positive_pool_size > sampled_positive_size:
                    extra_positive_size = min(remaining_to_fill, replay_positive_pool_size - sampled_positive_size)
                    sampled_positive_size += extra_positive_size
                    remaining_to_fill -= extra_positive_size
                if remaining_to_fill > 0 and replay_negative_pool_size > sampled_negative_size:
                    extra_negative_size = min(remaining_to_fill, replay_negative_pool_size - sampled_negative_size)
                    sampled_negative_size += extra_negative_size
                    remaining_to_fill -= extra_negative_size

                sampled_positive_groups = sample_grouped_indices(positive_sources, sampled_positive_size, replay_rng)
                sampled_negative_groups = sample_grouped_indices(negative_sources, sampled_negative_size, replay_rng)

                grouped_replay_indices = sampled_positive_groups + sampled_negative_groups

                if grouped_replay_indices:
                    X_replay_sampled, y_replay_sampled = materialize_replay_samples(
                        grouped_replay_indices,
                        train_feature_cache,
                        incremental_scaler,
                    )

                    if len(y_replay_sampled) > 0:
                        replay_shuffle_idx = replay_rng.permutation(len(y_replay_sampled))
                        X_replay_sampled = X_replay_sampled[replay_shuffle_idx]
                        y_replay_sampled = y_replay_sampled[replay_shuffle_idx]

                        replay_actual_positive_rate = float(np.mean(y_replay_sampled == REPLAY_POSITIVE_LABEL))
                        replay_class_weight_dict, replay_weight_fallback = compute_binary_class_weights(
                            y_replay_sampled,
                            classes=classes,
                            fallback_mode=REPLAY_WEIGHT_FALLBACK,
                        )
                        X_combined = np.concatenate([X_train_batch, X_replay_sampled], axis=0)
                        y_combined = np.concatenate([y_train_batch, y_replay_sampled], axis=0)
                        origin_is_replay_combined = np.concatenate(
                            [
                                np.zeros(len(y_train_batch), dtype=bool),
                                np.ones(len(y_replay_sampled), dtype=bool),
                            ]
                        )
                        replay_used_size = int(len(y_replay_sampled))
                    else:
                        X_combined = X_train_batch
                        y_combined = y_train_batch
                        origin_is_replay_combined = np.zeros(len(y_train_batch), dtype=bool)
                        replay_used_size = 0
                else:
                    X_combined = X_train_batch
                    y_combined = y_train_batch
                    origin_is_replay_combined = np.zeros(len(y_train_batch), dtype=bool)
                    replay_used_size = 0
            else:
                X_combined = X_train_batch
                y_combined = y_train_batch
                origin_is_replay_combined = np.zeros(len(y_train_batch), dtype=bool)

            combined_n_samples = len(X_combined)
            shuffle_idx = replay_rng.permutation(combined_n_samples)
            X_train_shuffled = X_combined[shuffle_idx]
            y_train_shuffled = y_combined[shuffle_idx]
            origin_is_replay_shuffled = origin_is_replay_combined[shuffle_idx]
            n_chunks = max(1, int(np.ceil(combined_n_samples / CHUNK_SIZE)))

            for chunk_idx in range(n_chunks):
                start_idx = chunk_idx * CHUNK_SIZE
                end_idx = min(start_idx + CHUNK_SIZE, combined_n_samples)
                X_chunk = X_train_shuffled[start_idx:end_idx]
                y_chunk = y_train_shuffled[start_idx:end_idx]
                origin_is_replay_chunk = origin_is_replay_shuffled[start_idx:end_idx]

                sample_weights_chunk = np.empty(len(y_chunk), dtype=np.float64)
                current_chunk_mask = ~origin_is_replay_chunk
                replay_chunk_mask = origin_is_replay_chunk
                if np.any(current_chunk_mask):
                    sample_weights_chunk[current_chunk_mask] = np.array(
                        [current_class_weight_dict[int(label)] for label in y_chunk[current_chunk_mask]],
                        dtype=np.float64,
                    )
                if np.any(replay_chunk_mask):
                    sample_weights_chunk[replay_chunk_mask] = np.array(
                        [replay_class_weight_dict[int(label)] for label in y_chunk[replay_chunk_mask]],
                        dtype=np.float64,
                    )

                if len(sample_weights_chunk) != len(y_chunk):
                    raise ValueError(
                        f'Sample weight length mismatch: weights={len(sample_weights_chunk)} vs labels={len(y_chunk)}'
                    )
                if np.any(~np.isfinite(sample_weights_chunk)):
                    raise ValueError('Encountered non-finite sample weights in training chunk.')
                if np.any(sample_weights_chunk <= 0):
                    raise ValueError('Encountered non-positive sample weights in training chunk.')

                model.partial_fit(X_chunk, y_chunk, classes=classes, sample_weight=sample_weights_chunk)

            current_mask = ~origin_is_replay_shuffled
            replay_mask = origin_is_replay_shuffled
            if np.any(current_mask):
                current_effective_mean_weight = float(
                    np.mean(
                        np.array([current_class_weight_dict[int(label)] for label in y_train_shuffled[current_mask]], dtype=np.float64)
                    )
                )
            if np.any(replay_mask):
                replay_effective_mean_weight = float(
                    np.mean(
                        np.array([replay_class_weight_dict[int(label)] for label in y_train_shuffled[replay_mask]], dtype=np.float64)
                    )
                )

            y_val_pred = model.predict(X_val_batch)
            y_val_proba = model.predict_proba(X_val_batch)[:, 1]
            val_pr_auc = average_precision_score(y_val_batch, y_val_proba) if len(np.unique(y_val_batch)) > 1 else np.nan

            if val_pr_auc > best_val_pr_auc + MIN_DELTA:
                best_val_pr_auc = val_pr_auc
                patience_counter = 0
                best_model_state = {
                    'coefs': [w.copy() for w in model.coefs_],
                    'intercepts': [b.copy() for b in model.intercepts_],
                    'n_layers_': model.n_layers_,
                    'n_outputs_': getattr(model, 'n_outputs_', None),
                    'out_activation_': getattr(model, 'out_activation_', None),
                }
            else:
                patience_counter += 1
                if patience_counter >= PATIENCE:
                    if best_model_state is not None:
                        model.coefs_ = [w.copy() for w in best_model_state['coefs']]
                        model.intercepts_ = [b.copy() for b in best_model_state['intercepts']]
                        model.n_layers_ = best_model_state['n_layers_']
                        if best_model_state['n_outputs_'] is not None:
                            model.n_outputs_ = best_model_state['n_outputs_']
                        if best_model_state['out_activation_'] is not None:
                            model.out_activation_ = best_model_state['out_activation_']
                    break

        y_train_pred = model.predict(X_train_batch)
        y_val_pred = model.predict(X_val_batch)
        y_val_proba = model.predict_proba(X_val_batch)[:, 1]

        train_acc = accuracy_score(y_train_batch, y_train_pred)
        train_prec = precision_score(y_train_batch, y_train_pred, zero_division=0)
        train_rec = recall_score(y_train_batch, y_train_pred, zero_division=0)
        train_f1 = f1_score(y_train_batch, y_train_pred, zero_division=0)

        val_acc = accuracy_score(y_val_batch, y_val_pred)
        val_prec = precision_score(y_val_batch, y_val_pred, zero_division=0)
        val_rec = recall_score(y_val_batch, y_val_pred, zero_division=0)
        val_f1 = f1_score(y_val_batch, y_val_pred, zero_division=0)
        if len(np.unique(y_val_batch)) > 1:
            val_roc_auc = roc_auc_score(y_val_batch, y_val_proba)
            val_pr_auc = average_precision_score(y_val_batch, y_val_proba)
        else:
            val_roc_auc = np.nan
            val_pr_auc = np.nan

        training_history['year'].append(year_val)
        training_history['train_accuracy'].append(train_acc)
        training_history['train_precision'].append(train_prec)
        training_history['train_recall'].append(train_rec)
        training_history['train_f1'].append(train_f1)
        training_history['val_accuracy'].append(val_acc)
        training_history['val_precision'].append(val_prec)
        training_history['val_recall'].append(val_rec)
        training_history['val_f1'].append(val_f1)
        training_history['val_roc_auc'].append(val_roc_auc)
        training_history['val_pr_auc'].append(val_pr_auc)
        training_history['replay_pool_size'].append(int(replay_pool_size))
        training_history['replay_target_size'].append(int(replay_target_size))
        training_history['replay_used_size'].append(int(replay_used_size))
        training_history['replay_target_positive_rate'].append(float(TARGET_REPLAY_POSITIVE_RATE))
        training_history['replay_actual_positive_rate'].append(float(replay_actual_positive_rate) if replay_used_size > 0 else np.nan)
        training_history['weight_policy'].append(WEIGHT_POLICY_NAME)
        training_history['current_origin_weight'].append(np.nan)
        training_history['year_positive_rate'].append(float(year_positive_rate))
        training_history['target_positive_rate'].append(float(TARGET_REPLAY_POSITIVE_RATE))
        training_history['replay_multiplier_unclipped'].append(np.nan)
        training_history['replay_multiplier'].append(np.nan)
        training_history['replay_pool_positive_rate'].append(float(replay_pool_positive_rate) if replay_pool_size > 0 else np.nan)
        training_history['current_class_weight_0'].append(float(current_class_weight_dict[0]))
        training_history['current_class_weight_1'].append(float(current_class_weight_dict[1]))
        training_history['replay_class_weight_0'].append(float(replay_class_weight_dict[0]) if replay_used_size > 0 else np.nan)
        training_history['replay_class_weight_1'].append(float(replay_class_weight_dict[1]) if replay_used_size > 0 else np.nan)
        training_history['replay_weight_fallback'].append(replay_weight_fallback)

        year_model_path = models_dir / model_file_template.format(year=year_val)
        with open(year_model_path, 'wb') as f:
            pickle.dump(model, f)

        completed_years.add(year_val)
        completion_status['completed_years'][ratio_key] = sorted(list(completed_years))
        all_training_histories[ratio_key] = training_history.copy()
        save_all_training_histories(ALL_HISTORIES_FILE, all_training_histories)
        save_completion_status(COMPLETION_STATUS_FILE, completion_status)

        print(
            f'[{ratio_key}] Year {year_val}: Train F1={train_f1:.3f}, Val F1={val_f1:.3f}, Val PR-AUC={val_pr_auc:.3f}, '
            f'replay_used={replay_used_size:,}/{replay_pool_size:,}, replay_pos_rate={replay_actual_positive_rate:.2%}, '
            f'year_pos_rate={year_positive_rate:.2%}, replay_w0={replay_class_weight_dict[0]:.3f}, replay_w1={replay_class_weight_dict[1]:.3f}'
        )
        append_training_log(
            TRAINING_LOG_FILE,
            f'[{ratio_key}] completed year {year_val} with replay_used={replay_used_size}/{replay_pool_size}, '
            f'replay_pos_rate={replay_actual_positive_rate:.4f}, year_pos_rate={year_positive_rate:.4f}, '
            f'replay_pool_pos_rate={replay_pool_positive_rate:.4f}, current_w0={current_class_weight_dict[0]:.4f}, '
            f'current_w1={current_class_weight_dict[1]:.4f}, replay_w0={replay_class_weight_dict[0]:.4f}, '
            f'replay_w1={replay_class_weight_dict[1]:.4f}, replay_weight_fallback={replay_weight_fallback}, '
            f'current_effective_mean_weight={current_effective_mean_weight:.4f}, replay_effective_mean_weight={replay_effective_mean_weight:.4f}.'
        )

    if hasattr(incremental_scaler, 'mean_'):
        final_scaler_path = models_dir / final_scaler_filename
        with open(final_scaler_path, 'wb') as f:
            pickle.dump(incremental_scaler, f)
        print(f'[{ratio_key}] Final incremental scaler saved: {final_scaler_path.name}')

    final_model_path = models_dir / final_model_filename
    with open(final_model_path, 'wb') as f:
        pickle.dump(model, f)
    print(f'[{ratio_key}] Final model saved: {final_model_path.name}')

    ratio_history_df = pd.DataFrame(training_history).sort_values('year').reset_index(drop=True)
    ratio_history_path = models_dir / history_filename
    ratio_history_df.to_csv(ratio_history_path, index=False)
    print(f'[{ratio_key}] History saved: {ratio_history_path}')

    if set(all_target_year_values).issubset(completed_years):
        if ratio_key not in completion_status.get('completed_ratios', []):
            completion_status.setdefault('completed_ratios', []).append(ratio_key)
        save_completion_status(COMPLETION_STATUS_FILE, completion_status)
        append_training_log(TRAINING_LOG_FILE, f'[{ratio_key}] marked as fully completed.')

    all_training_histories[ratio_key] = training_history.copy()
    save_all_training_histories(ALL_HISTORIES_FILE, all_training_histories)

combined_history_frames = []
for ratio_key, history_dict in all_training_histories.items():
    if not isinstance(history_dict, dict):
        continue
    if len(history_dict.get('year', [])) == 0:
        continue
    df_ratio = pd.DataFrame(history_dict).sort_values('year').reset_index(drop=True)
    replay_ratio = float(ratio_key.split('_')[1])
    df_ratio['ratio_key'] = ratio_key
    df_ratio['replay_ratio'] = replay_ratio
    combined_history_frames.append(df_ratio)

if combined_history_frames:
    combined_history_df = pd.concat(combined_history_frames, ignore_index=True)
    combined_history_df = combined_history_df.sort_values(['replay_ratio', 'year']).reset_index(drop=True)
    combined_history_path = Path('.') / COMBINED_HISTORY_FILENAME
    combined_history_df.to_csv(combined_history_path, index=False)
    print(f'Combined history saved: {combined_history_path}')
    print(f'Combined rows: {len(combined_history_df):,}')
else:
    print('No history data available to write combined history.')

save_completion_status(COMPLETION_STATUS_FILE, completion_status)
save_all_training_histories(ALL_HISTORIES_FILE, all_training_histories)
append_training_log(TRAINING_LOG_FILE, 'Training run completed.')


Checkpoint directory: C:\Users\bartu\Desktop\Fonda-scikit - Git\training_checkpoints_mlp_experience_replay_PR_0.15
Replay ratios: [0.2, 0.3, 0.4, 0.5]
Replay target positive rate: 15.00% (PR_0.15)
Weight policy: post_sampling_replay_class_weights_v1, replay_weight_fallback=smoothed_single_class
[RR_0.2] Model output directory: C:\Users\bartu\Desktop\Fonda-scikit - Git\models_mlp_prevyears_monthly_features_incremental_scaler_experience_replay_RR_0.2_PR_0.15
[RR_0.2] Years to train: 1..6 (year 0 skipped)


RR_0.2 by year:   0%|          | 0/6 [00:00<?, ?it/s]

[RR_0.2] Year 2017: Train F1=0.147, Val F1=0.135, Val PR-AUC=0.181, replay_used=0/0, replay_pos_rate=nan%, year_pos_rate=1.85%, replay_w0=nan, replay_w1=nan


TypeError: append_training_log() takes 2 positional arguments but 5 were given

## 5. Save Training History

In [ ]:
import json

TARGET_REPLAY_POSITIVE_RATE = 0.15
TARGET_POSITIVE_RATE_TOKEN = f'PR_{TARGET_REPLAY_POSITIVE_RATE:.2f}'

CHECKPOINT_DIR = Path('.') / f'training_checkpoints_mlp_experience_replay_{TARGET_POSITIVE_RATE_TOKEN}'
ALL_HISTORIES_FILE = CHECKPOINT_DIR / 'mlp_replay_all_training_histories.pkl'
COMPLETION_STATUS_FILE = CHECKPOINT_DIR / 'mlp_replay_completion_status.json'
COMBINED_HISTORY_FILENAME = f'mlp_classifier_history_prevyears_monthly_features_incremental_scaler_experience_replay_{TARGET_POSITIVE_RATE_TOKEN}_all_ratios.csv'

if ALL_HISTORIES_FILE.exists():
    with open(ALL_HISTORIES_FILE, 'rb') as f:
        all_training_histories = pickle.load(f)
else:
    all_training_histories = {}

if COMPLETION_STATUS_FILE.exists():
    with open(COMPLETION_STATUS_FILE, 'r', encoding='utf-8') as f:
        completion_status = json.load(f)
else:
    completion_status = {'completed_ratios': [], 'completed_years': {}}

summary_rows = []
for ratio_key in sorted(all_training_histories.keys()):
    history_dict = all_training_histories[ratio_key]
    n_rows = len(history_dict.get('year', []))
    last_year = history_dict['year'][-1] if n_rows > 0 else np.nan
    last_val_f1 = history_dict['val_f1'][-1] if n_rows > 0 else np.nan
    last_val_pr_auc = history_dict['val_pr_auc'][-1] if n_rows > 0 else np.nan
    is_completed = ratio_key in completion_status.get('completed_ratios', [])
    summary_rows.append(
        {
            'ratio_key': ratio_key,
            'rows': n_rows,
            'last_year': last_year,
            'last_val_f1': last_val_f1,
            'last_val_pr_auc': last_val_pr_auc,
            'completed': is_completed,
        }
    )

summary_df = pd.DataFrame(summary_rows).sort_values('ratio_key').reset_index(drop=True)
print('Per-ratio training summary:')
display(summary_df)

combined_history_path = Path('.') / COMBINED_HISTORY_FILENAME
if combined_history_path.exists():
    combined_df = pd.read_csv(combined_history_path)
    print(f'Combined history file: {combined_history_path}')
    print(f'Rows: {len(combined_df):,}')
    display(combined_df.tail())
else:
    print(f'Combined history file not found: {combined_history_path}')

Per-ratio training summary:


,ratio_key,rows,last_year,last_val_f1,last_val_pr_auc,completed
0,RR_0.2,6,2022,0.164572,0.351865,True
1,RR_0.3,6,2022,0.160873,0.354822,True
2,RR_0.4,6,2022,0.161542,0.353989,True
3,RR_0.5,6,2022,0.154856,0.344977,True


Combined history file: mlp_classifier_history_prevyears_monthly_features_incremental_scaler_experience_replay_PR_0.10_all_ratios.csv
Rows: 24


,year,train_accuracy,train_precision,train_recall,train_f1,val_accuracy,val_precision,val_recall,val_f1,val_roc_auc,val_pr_auc,replay_pool_size,replay_target_size,replay_used_size,replay_target_positive_rate,replay_actual_positive_rate,ratio_key,replay_ratio
19,2018,0.871580,0.117729,0.808719,0.205537,0.863646,0.115404,0.774056,0.200861,0.902588,0.428794,5575648,2793403,2793403,0.1,0.036855,RR_0.5,0.5
20,2019,0.883407,0.123341,0.778854,0.212957,0.887976,0.114077,0.757992,0.198309,0.904190,0.418609,11162455,2793385,2793385,0.1,0.077937,RR_0.5,0.5
21,2020,0.799854,0.073832,0.844060,0.135787,0.806261,0.067952,0.821883,0.125525,0.900113,0.442260,16749226,2793696,2793696,0.1,0.100000,RR_0.5,0.5
22,2021,0.818438,0.067059,0.846311,0.124271,0.811549,0.064873,0.827551,0.120315,0.904694,0.420090,22336618,2793427,2793427,0.1,0.100000,RR_0.5,0.5
23,2022,0.736656,0.094649,0.886399,0.171035,0.732122,0.085175,0.851333,0.154856,0.879281,0.344977,27923473,2796040,2796040,0.1,0.100000,RR_0.5,0.5
